In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, ElasticNet, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import  ColumnTransformer
from sklearn.compose import make_column_selector
from sklearn.impute import SimpleImputer 
import warnings
warnings.filterwarnings('ignore')

In [33]:
train = pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\Day3\\playground-series-s5e1\\train.csv")
train

,id,date,country,store,product,num_sold
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.0
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.0
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.0
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.0
...,...,...,...,...,...,...
230125,230125,2016-12-31,Singapore,Premium Sticker Mart,Holographic Goose,466.0
230126,230126,2016-12-31,Singapore,Premium Sticker Mart,Kaggle,2907.0
230127,230127,2016-12-31,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.0
230128,230128,2016-12-31,Singapore,Premium Sticker Mart,Kerneler,1242.0


In [34]:
train.isnull().sum()

id             0
date           0
country        0
store          0
product        0
num_sold    8871
dtype: int64

In [36]:
train = train.dropna(subset=['num_sold'])
train

,id,date,country,store,product,num_sold
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.0
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.0
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.0
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.0
5,5,2010-01-01,Canada,Stickers for Less,Holographic Goose,300.0
...,...,...,...,...,...,...
230125,230125,2016-12-31,Singapore,Premium Sticker Mart,Holographic Goose,466.0
230126,230126,2016-12-31,Singapore,Premium Sticker Mart,Kaggle,2907.0
230127,230127,2016-12-31,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.0
230128,230128,2016-12-31,Singapore,Premium Sticker Mart,Kerneler,1242.0


In [37]:

X = train.drop(['id', 'date', 'num_sold'], axis=1)
y = train['num_sold']

dum_train = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26)

In [38]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")

trans = ColumnTransformer(
    transformers=[("OHE", ohe,make_column_selector(dtype_include=object))],remainder="passthrough",
    verbose_feature_names_out=False).set_output(transform="pandas")

In [39]:
X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)

In [40]:
# Linear Regression
lr=LinearRegression()
lr.fit(X_trn_ohe,y_train)
y_pred =lr.predict(X_tst_ohe)
print("R2 =",r2_score(y_test,y_pred))

R2 = 0.7896910334776606


In [41]:
# lasso Regression
alphas=np.linspace(0.001, 5)
scores=[]

for a in alphas:
    lasso = Lasso(alpha = a)
    lasso.fit(X_trn_ohe, y_train)
    y_pred = lasso.predict(X_tst_ohe)
    scores.append([a, r2_score(y_test, y_pred)])
    
df_scores = pd.DataFrame(scores, columns=['alpha','score'])
df_scores.sort_values('score', ascending=False).head()

,alpha,score
0,0.001000,0.789691
1,0.103020,0.789684
2,0.205041,0.789668
3,0.307061,0.789644
4,0.409082,0.789615


In [42]:
# Ridge Regression
alphas=np.linspace(0.001, 5)
scores=[]

for a in alphas:
    ridge = Ridge(alpha = a)
    ridge.fit(X_trn_ohe, y_train)
    y_pred = ridge.predict(X_tst_ohe)
    scores.append([a, r2_score(y_test, y_pred)])
    
df_scores = pd.DataFrame(scores, columns=['alpha','score'])
df_scores.sort_values('score', ascending=False).head()

,alpha,score
0,0.001000,0.789691
1,0.103020,0.789691
2,0.205041,0.789691
3,0.307061,0.789691
4,0.409082,0.789691


In [44]:
alphas=np.linspace(0.001, 5, 20)
ratio=np.linspace(0, 1, 20)
scores=[]

for a in alphas:
    for r in ratio:
        e1 = ElasticNet(alpha = a, l1_ratio = r)
        e1.fit(X_trn_ohe, y_train)
        y_pred = e1.predict(X_tst_ohe)
        scores.append([a, r, r2_score(y_test, y_pred)])
        
df_scores = pd.DataFrame(scores, columns=['alpha', 'score', 'l1_ratio'])
df_scores.sort_values('score', ascending=False).head()

,alpha,score,l1_ratio
19,0.001000,1.0,0.789691
399,5.000000,1.0,0.780680
159,1.842737,1.0,0.788448
239,2.895158,1.0,0.786654
79,0.790316,1.0,0.789448


# Inferencing

In [ ]:
tst = pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\Day3\\playground-series-s5e1\\train.csv")
tst

In [ ]:
X_ohe=trans.fit_transform(X)
bm = Ridge(alpha= 0.001000)
bm.fit(X_ohe, y)

In [ ]:
test_ohe=trans.transform(tst)
y_pred=bm.predict(test_ohe)
y_pred

In [ ]:
ss = pd.read_csv("sample_submission.csv")
ss['num_sold']=y_pred
ss.to_csv("sbt_log_18_4.csv", index=False)
ss